This notebook is for modeling and evaluating span identification from the SemEval dataset. This is the first filter for propaganda, so we only want to filter out what we are confident is not propaganda (so high-sensitivity/high-recall). Then downstream, let the technique classification (TC) model handle the precision and pruning.

Best performance of RoBERTa model (training to optimize F2 score, then adjusted threshold to attempt to reach 0.9 recall) using BIO tagging technique rather than multi-class approach:
### LR=2.5e-05, WD=0.15

| Epoch | Training Loss | Validation Loss | Precision | Recall | F1 Score | F2 Score | Threshold |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 0.406731 | 0.325883 | 0.167925 | 0.798293 | 0.277480 | 0.455965 | 0.100000 |
| 2 | 0.336962 | 0.317658 | 0.222364 | 0.678492 | 0.334953 | 0.481114 | 0.100000 |
| 3 | 0.314832 | 0.330318 | 0.241480 | 0.668884 | 0.354851 | 0.494010 | 0.100000 |
| 4 | 0.242153 | 0.341557 | 0.274098 | 0.556944 | 0.367388 | 0.461664 | 0.100000 |
| 5 | 0.208649 | 0.352840 | 0.289933 | 0.529665 | 0.374739 | 0.454504 | 0.100000 |
| 6 | 0.165558 | 0.371711 | 0.297578 | 0.533293 | 0.382000 | 0.460361 | 0.100000 |
| 7 | 0.158078 | 0.382776 | 0.305843 | 0.495868 | 0.378335 | 0.441060 | 0.100000 |
| 8 | 0.133436 | 0.386377 | 0.294015 | 0.544648 | 0.381881 | 0.465317 | 0.100000 |

To implement the Claimify approach rather than the binary "Propaganda vs. Not" approach, we are treated this as a Categorical Span Identification task. The idea is that not all propaganda is created equal, and different techniques are very different from each other linguistically.

In [1]:
import os
import json
import torch
from torch import nn
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from sklearn.metrics import precision_recall_fscore_support
from datasets import Dataset
from accelerate.state import AcceleratorState
import zipfile
import shutil
import gdown
import evaluate

In [2]:
AcceleratorState._reset_state()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
#Set up paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "semeval_roberta_scanner"

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [5]:
def setup_models(file_id, target_path):
    zip_temp = target_path.with_suffix(".zip")
    if not (target_path / "model.safetensors").exists() and not (target_path / "pytorch_model.bin").exists():
        print(f"Model not found. Downloading from Google Drive...")
        target_path.parent.mkdir(exist_ok=True, parents=True)
        url = f'https://drive.google.com/uc?id={file_id}'
        try:
            gdown.download(url, str(zip_temp), quiet=False)
            with zipfile.ZipFile(zip_temp, 'r') as zip_ref:
                zip_ref.extractall(target_path.parent)
            os.remove(zip_temp)
            return True
        except Exception as e:
            print(f"Download failed: {e}")
            return False
    return True

#Identify if model exists
model_exists = setup_models('1', MODEL_DIR)

#model_exists = setup_models('13SC1hSucXUmTtdNKRK8TgytdDNJ4yqXi', MODEL_DIR)

Model not found. Downloading from Google Drive...
Download failed: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1

but Gdown can't. Please check connections and permissions.


In [6]:
#Load article-level span identification data
df_si = pd.read_csv(DATA_DIR / "semeval_si_cleaned.csv")
df_si['propaganda_offsets'] = df_si['propaganda_offsets'].apply(json.loads)
print(f"Loaded {len(df_si)} articles.")
df_si.head()

Loaded 357 articles.


,article_id,text,propaganda_offsets
0,111111111,Next plague outbreak in Madagascar could be 's...,"[[265, 323], [1795, 1935], [149, 157], [1069, ..."
1,111111112,US bloggers banned from entering UK\n\nTwo pro...,"[[191, 219], [476, 556], [785, 798], [958, 101..."
2,111111113,Kate Steinle's death at the hands of a Mexican...,"[[1396, 1430], [3082, 3099], [3828, 3985], [36..."
3,111111114,U.S. judge frees Indonesian immigrant held by ...,"[[1705, 1824]]"
4,111111115,Here are all the sexual misconduct accusations...,"[[658, 700], [1870, 1893], [1655, 1745], [2389..."


In [7]:
#Initialize the model tokenizer
##Tried "bert-base-uncased", Best F1 score after 3 epochs: 0.26392
##"microsoft/deberta-v3-small", After 3 epoches, still getting gradient explosion every time, even after adjusting hyperparameters
raw_dataset = Dataset.from_pandas(df_si)
tokenizer = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

In [22]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_inputs.pop("offset_mapping")
    labels = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        article_spans = examples["propaganda_offsets"][sample_idx]
        doc_labels = []
        for start, end in offsets:
            if start == end == 0:
                doc_labels.append(-100)
                continue
            is_prop = any(s <= start < e or s < end <= e for s, e in article_spans)
            doc_labels.append(1 if is_prop else 0)
        labels.append(doc_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

#Tokenize and split data
tokenized_datasets = raw_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=raw_dataset.column_names).train_test_split(test_size=0.2, seed=42)

Map:   0%|          | 0/357 [00:00<?, ? examples/s]

In [23]:
#Tried without weighting before and was quickly overfitting, so weight now
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        #Prioritize Recall: Propaganda classes (1, 2) weighted 20x more than background (0)
        weights = torch.tensor([1.0, 20.0, 20.0], device = model.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [24]:
def compute_metrics(p):
    logits, labels = p
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    prop_probs = probs[:, :, 1] + probs[:, :, 2] # Sum of B and I labels

    y_true = labels.flatten()
    mask = y_true != -100
    y_true_clean = (y_true[mask] > 0).astype(int)
    prop_probs_clean = prop_probs.flatten()[mask]

    #Search from 0.1 to 0.50
    thresholds = np.arange(0.1, 0.51, 0.02)
    best_metrics = {"precision": 0, "recall": 0, "f1": 0, "threshold": 0.01}

    target_recall = 0.90
    found_target = False
    best_p_at_target = -1

    for threshold in thresholds:
        y_pred = (prop_probs_clean >= threshold).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true_clean, y_pred, average='binary', zero_division=0)

        if r >= target_recall:
            found_target = True
            if p > best_p_at_target:
                best_p_at_target = p
                best_metrics = {"precision": p, "recall": r, "f1": f1, "threshold": threshold}
        elif not found_target:
            # Fallback: if we can't hit 0.90, just keep the highest recall version
            if r > best_metrics["recall"]:
                best_metrics = {"precision": p, "recall": r, "f1": f1, "threshold": threshold}

    return best_metrics

In [25]:
#Initialize model - Load from local if exists, else from checkpoint
if (MODEL_DIR / "config.json").exists():
    print(f"Loading existing trained model from: {MODEL_DIR}")
    model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
    model_already_trained = True
else:
    print(f"No existing model found. Initializing from: {"roberta-base"}")
    model = AutoModelForTokenClassification.from_pretrained("roberta-base", num_labels=3)
    model_already_trained = False

model.to(device)


No existing model found. Initializing from: roberta-base


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [26]:
#Set up training arguments with optimized hyperparameters
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2.5e-05,
    per_device_train_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.15,
    logging_steps=5,
    metric_for_best_model="f2",
    greater_is_better=True,
    dataloader_pin_memory=False,
    disable_tqdm=False,
    report_to="none",
    load_best_model_at_end=True
)

In [27]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
#Train only if we didn't load a local model
if not model_already_trained:
    print("Starting training process...")
    trainer.train()
    trainer.save_model(MODEL_DIR)
    tokenizer.save_pretrained(MODEL_DIR)
    print(f"Model trained and saved to {MODEL_DIR}")
else:
    print("Model loaded from disk. Skipping training.")

Starting training process...


Epoch,Training Loss,Validation Loss


In [ ]:
#Evaluate performance on the test dataset
trainer.remove_callback(NotebookProgressCallback)
test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])

print("\n" + "="*30)
print("FINAL MODEL PERFORMANCE")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"F2 Score:  {test_results['eval_f2_score']:.4f}")
print("="*30)

In [ ]:
#Compare results on train vs. test sets to ensure not overfitting
#Force evaluation on the Train set
train_results = trainer.evaluate(eval_dataset=tokenized_datasets["train"])
print(f"TRAIN Recall: {train_results['eval_recall']:.4f}")